# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI

from agents.deals_async import AsyncScrapedDeal
from agents.deals_common import DealSelection

In [2]:
# Initialize and constants

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [3]:
# deals = ScrapedDeal.fetch(show_progress=True)
import nest_asyncio
nest_asyncio.apply()


In [4]:
deals = await AsyncScrapedDeal.fetch_async(show_progress=True)

📦 Fetching All Feeds: 100%|██████████████████████████████████████████████████████████████████████| 9/9 [00:58<00:00,  6.54s/it]


In [5]:
len(deals)

90

In [6]:
deals[44].describe()

"Title: Whiffle 1080p Wireless Doorbell Camera for $33 + free shipping\nDetails: That's a savings of $27. Buy Now at ANNKE\nFeatures: \nURL: https://www.dealnews.com/Whiffle-1080-p-Wireless-Doorbell-Camera-for-33-free-shipping/21742097.html?iref=rss-c196"

In [7]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [8]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [9]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: iCharge Nano 2000mAh Keychain USB-C Power Bank for $17 + $3 s&h
Details: Save $33 on this charger for your phone, smart watch, or headphones. Buy Now at StackSocial
Features: delivers 25-35% extra power secure magnetic alignment for Apple Watch TSA-friendly design
URL: https://www.dealnews.com/iCharge-Nano-2000-m-Ah-Keychain-USB-C-Power-Bank-for-17-3-s-h/21742096.ht

In [10]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    result = completion.choices[0].message.parsed
    return result

In [11]:
result = get_recommendations()

In [12]:
len(result.deals)

5

In [13]:
result.deals[1]

Deal(product_description='The Weather Hi-Def Radar Storm Watch Plus application offers a comprehensive lifetime subscription that includes real-time radar images, future forecasts, and severe weather alerts. Designed for new users, this service is invaluable for keeping informed about the weather with interactive map layers and timely notifications, ensuring you stay safe and prepared.', price=28.0, url='https://www.dealnews.com/Weather-Hi-Def-Radar-Storm-Watch-Plus-Lifetime-subscription-for-28/21742069.html?iref=rss-c142')

In [ ]:
from agents.scanner_agent_async import ScannerAgentAsync

## ScannerAsyncAgent is using 'gpt-4o-mini'

In [ ]:
agent = ScannerAgentAsync(show_progress=True)
result = agent.scan()

📦 Fetching All Feeds: 100%|██████████████████████████████████████████████████████████████████████| 9/9 [00:46<00:00,  5.22s/it]


In [16]:
result.deals

[Deal(product_description='The iCharge Nano is a portable 2000mAh keychain USB-C power bank designed for convenience and efficiency. It provides 25-35% extra power for charging your phone, smart watch, or headphones on the go. Featuring secure magnetic alignment specifically for Apple Watch, it ensures a reliable connection. Its TSA-friendly design makes it perfect for travel, easily fitting into your bag without hassle.', price=17.0, url='https://www.dealnews.com/iCharge-Nano-2000-m-Ah-Keychain-USB-C-Power-Bank-for-17-3-s-h/21742096.html?iref=rss-c142'),
 Deal(product_description='The Certified Refurb LG S40T is a 2.1-channel soundbar that comes with a wireless subwoofer, providing a cinematic audio experience in your home. It features Dolby Digital and DTS Digital compatibility for immersive sound, and Dolby Audio Clear Voice Plus technology enhances dialogue clarity. The built-in 3-band equalizer allows for sound customization, making it a perfect fit for movie nights or parties.', 

In [17]:
print(result.deals[0])

product_description='The iCharge Nano is a portable 2000mAh keychain USB-C power bank designed for convenience and efficiency. It provides 25-35% extra power for charging your phone, smart watch, or headphones on the go. Featuring secure magnetic alignment specifically for Apple Watch, it ensures a reliable connection. Its TSA-friendly design makes it perfect for travel, easily fitting into your bag without hassle.' price=17.0 url='https://www.dealnews.com/iCharge-Nano-2000-m-Ah-Keychain-USB-C-Power-Bank-for-17-3-s-h/21742096.html?iref=rss-c142'


In [18]:
print(type(result))

<class 'agents.deals_common.DealSelection'>
